# Lecture 12: Non-Linear Programming — Principles

---

```{note}
Lectures 1–11 built a complete toolkit for **linear** optimisation: the objective function and every constraint were straight lines (or hyperplanes). Many real transportation problems — signal timing, congestion pricing, facility siting, vehicle routing — are not linear: costs rise non-proportionally with volume, constraints are products or ratios of decision variables, and the feasible region is no longer a convex polygon. This lecture introduces the landscape of **Non-Linear Programming (NLP)**: what makes a problem non-linear, how non-linearity changes the geometry and the solution strategy, and which families of NLP methods we will study in Lectures 13–22.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Recognise the structural differences between linear and non-linear programs, and classify NLP instances by problem type (unconstrained/constrained, convex/non-convex).
2. Identify local and global optima geometrically and analytically, and explain why LP methods fail in non-linear settings.
3. Map real transportation optimisation problems to the appropriate NLP framework and solver family covered in this module.

**Prerequisites**: LP Formulation (Lecture 3), Graphical Solution Method (Lecture 4), Simplex Method (Lecture 7), Duality (Lecture 9)

**Estimated time**: 90 minutes (including in-class exercises)

---

## Why Non-Linear Programming?

Every LP we solved in Lectures 3–11 rested on a silent assumption: the relationships between decision variables are proportional and additive. Doubling the number of buses doubles the cost. Sending twice as much freight doubles fuel use. Adding one more worker doubles output. In practice this linearity assumption breaks down in three common ways.

**Volume effects.** Congestion on a road makes travel time rise faster than linearly with traffic flow. The Bureau of Public Roads (BPR) formula — used in traffic assignment across NHAI's corridor studies — says:

$$t(v) = t_0 \left[1 + \alpha \left(\frac{v}{c}\right)^\beta\right], \qquad \alpha = 0.15,\; \beta = 4$$

where $t_0$ is free-flow travel time, $v$ is volume, and $c$ is capacity. The $(\cdot)^4$ term makes total travel time a degree-5 polynomial in $v$ — nothing an LP can represent.

**Fixed and setup costs.** Opening a new freight terminal in Nagpur or Chennai incurs a fixed cost regardless of volume — a discontinuity (step function) that violates LP's linearity requirement.

**Interaction effects.** The fuel consumption of a truck on a hilly highway depends on both its load *and* its speed simultaneously — a product of two decision variables — making the objective non-linear.

> **Key question**: once we leave the land of linear programs, what tools replace the Simplex method?

---

## Standard Form

A **Non-Linear Program** is an optimisation problem of the form:

$$
\begin{aligned}
\min_{x} \quad & f(x) \\
\text{subject to} \quad & g_i(x) \leq 0, \qquad i = 1, \ldots, m \\
& x \in \mathbb{R}^n
\end{aligned}
$$

where at least one of $f$ or $g_i$ is **non-linear** in $x$. Equality constraints $g_j(x) = 0$ are handled exactly as in LP (Lecture 3) — converted to a pair $g_j(x) \leq 0$ and $-g_j(x) \leq 0$ — so no separate equality row is needed.

**Notation used throughout Module 2**

| Symbol | Meaning |
|--------|---------|
| $x = (x_1, \ldots, x_n)^\top$ | Vector of $n$ decision variables |
| $f : \mathbb{R}^n \to \mathbb{R}$ | Objective function |
| $g_i : \mathbb{R}^n \to \mathbb{R}$ | Constraint function ($\leq 0$ form; equalities handled by conversion) |
| $\nabla f(x)$ | Gradient of $f$ at $x$ — vector of first partial derivatives |
| $\nabla^2 f(x)$ | Hessian of $f$ at $x$ — matrix of second partial derivatives |
| $\mathcal{F}$ | Feasible region: $\{x : g_i(x) \leq 0 \; \forall\, i\}$ |

### Relation to LP

The LP studied in Lectures 1–11 is the special case where all functions are linear. In compact matrix–vector notation (Lecture 3, $\mathbf{c} \in \mathbb{R}^n$, $\mathbf{A} \in \mathbb{R}^{m \times n}$, $\mathbf{b} \in \mathbb{R}^m$):

$$\max_{\mathbf{x} \geq \mathbf{0}} \quad \mathbf{c}^\top \mathbf{x} \qquad \text{subject to} \qquad \mathbf{A}\mathbf{x} \leq \mathbf{b}$$

This is the NLP general form specialised to $f(\mathbf{x}) = -\mathbf{c}^\top \mathbf{x}$ (linear objective, negated for $\min$) and $g_i(\mathbf{x}) = \mathbf{A}_i \mathbf{x} - b_i \leq 0$ (linear constraints, where $\mathbf{A}_i$ is the $i$-th row of $\mathbf{A}$). Everything in Module 2 reduces to Module 1 when all functions are linear. The familiar LP optimality conditions (shadow prices, reduced costs, duality) become special cases of the NLP conditions (KKT multipliers, Lagrangian duality) we derive in Lectures 15–16.

---

## Taxonomy

Not all non-linear programs are equally hard. The two most important structural properties are: **(a) constrained or unconstrained?** and **(b) convex or non-convex?**

### Axis 1 — Constrained vs Unconstrained

**Unconstrained NLP**: no inequality or equality constraints; $\mathcal{F} = \mathbb{R}^n$.

$$\min_{x \in \mathbb{R}^n} \; f(x)$$

A necessary condition for $x^*$ to be a local minimum: $\nabla f(x^*) = \mathbf{0}$ (zero gradient — the NLP analogue of reduced costs all being non-negative in the Simplex method). Sufficiency requires the Hessian $\nabla^2 f(x^*)$ to be positive semi-definite.

**Constrained NLP**: at least one active constraint narrows $\mathcal{F}$. The Simplex method exploited constraint geometry (vertices of a polytope) to find the optimum. For NLP, constraints interact with the objective through **Lagrange multipliers** and **KKT conditions** (Lecture 15).

### Axis 2 — Convex vs Non-Convex

This is the most consequential distinction in NLP.

**Convex program**: $f$ is convex *and* $\mathcal{F}$ is a convex set (all $g_i$ are convex, all $h_j$ are linear).

$$\text{A function } f \text{ is convex if } f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda) f(y) \quad \forall x,y,\; \lambda \in [0,1]$$

The decisive property: **every local minimum of a convex program is also a global minimum**. This is the NLP analogue of LP's property that every basic feasible solution at a vertex is a candidate for the global optimum. Lecture 14 develops convexity theory in full.

**Non-convex program**: $f$ or $\mathcal{F}$ (or both) are non-convex. Local minima may not be global — gradient-based algorithms can get trapped.

### Summary Table

| | Unconstrained | Constrained |
|---|---|---|
| **Convex** | Easiest class; gradient = 0 is sufficient | LP is a special case; KKT conditions are sufficient |
| **Non-convex** | Multiple local minima; global search hard | Most general and hardest class (TSP, VRP live here) |

---

## Local vs Global Optima

The single most important conceptual shift from LP to NLP is the distinction between **local** and **global** optima.

**Definitions**

A point $x^*$ is a **local minimum** of $f$ if there exists $\epsilon > 0$ such that:
$$f(x^*) \leq f(x) \quad \forall\, x \in \mathcal{F} \text{ with } \|x - x^*\| < \epsilon$$

A point $x^*$ is a **global minimum** of $f$ on $\mathcal{F}$ if:
$$f(x^*) \leq f(x) \quad \forall\, x \in \mathcal{F}$$

Every global minimum is a local minimum. The converse is **false** in general (but true for convex programs — Lecture 14).

>**LP never had this problem!** In LP, the feasible region $\mathcal{F}$ is a convex polytope and the objective is linear — the surface is a flat hyperplane. Every local optimum is automatically global. The Simplex method exploits this by searching only the *vertices* of $\mathcal{F}$; there is no risk of getting trapped. In NLP, the objective surface can have hills and valleys. A gradient-descent algorithm that starts in one valley will converge to the bottom of that valley (a local minimum) and never know whether a deeper valley exists elsewhere.

The code cell below renders this geometry visually — a 1-D example that makes the distinction concrete.

---

## Optimality Conditions

### Unconstrained Case

For $\min f(x)$ with $x \in \mathbb{R}^n$:

**First-Order Necessary Condition (FONC)**: If $x^*$ is a local minimum and $f$ is differentiable, then:
$$\nabla f(x^*) = \mathbf{0}$$

A point satisfying the FONC is called a **stationary point** — it could be a minimum, maximum, or saddle point.

**Second-Order Sufficient Condition (SOSC)**: If $\nabla f(x^*) = \mathbf{0}$ and $\nabla^2 f(x^*)$ is **positive definite** (all eigenvalues $> 0$), then $x^*$ is a strict local minimum.

```{tip}
**Positive definiteness check (2×2 case)**: For a $2 \times 2$ Hessian
$H = \begin{pmatrix} a & b \\ b & d \end{pmatrix}$, positive definiteness requires $a > 0$ and $\det(H) = ad - b^2 > 0$.
This is the 2-variable version you will use most often in hand calculations.
```

### Constrained Case

For the constrained NLP, stationarity of $f$ alone is not enough — the feasible boundary may be active at the optimum. The correct condition involves the **Lagrangian function**:

$$\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^{m} \lambda_i\, g_i(x)$$

The **Karush-Kuhn-Tucker (KKT) conditions** — derived fully in Lecture 15 — require $\nabla_x \mathcal{L} = \mathbf{0}$ along with complementary slackness conditions on $\lambda_i$. They generalise the LP optimality condition (all reduced costs $\geq 0$) to the non-linear, constrained setting.

### Connection to LP Duality

In LP, the dual variables (shadow prices) of Lecture 9 are exactly the Lagrange multipliers of the LP's Lagrangian. The strong duality theorem ($z^* = w^*$) is the LP special case of a more general NLP duality result (Lecture 16). The entire shadow-price interpretation you built in Lecture 8 carries over — generalised to KKT multipliers.

---

## Why LP Methods Fail for NLP

The Simplex method's power comes from three facts that are unique to LP:

1. **The feasible region is a convex polytope.** Every corner (vertex) is a candidate optimal solution, and the Simplex method hops between adjacent vertices along improving edges.

2. **The objective is linear.** The gradient $\nabla f = \mathbf{c}$ is constant — it never changes direction, so the method always knows which way is "downhill."

3. **Optimality is global.** LP's strong duality theorem guarantees that when Simplex stops (all reduced costs $\geq 0$), the current vertex is globally optimal — no other point anywhere in $\mathcal{F}$ can do better.

None of these three properties holds for a general NLP:

| LP property | NLP breakdown |
|-------------|--------------|
| Polytope feasible region | $\mathcal{F}$ can be non-convex (curved constraints), disconnected, or unbounded |
| Constant gradient | $\nabla f(x)$ changes with $x$ — Simplex's "improving direction" no longer makes sense |
| Global = local | Gradient $= 0$ is only a *necessary* condition; a stationary point may be a saddle or local minimum far from the global optimum |
| Vertex search is finite | No finite set of "corner candidates" — a continuous, potentially infinite search |

The methods in Lectures 13–19 replace vertex-hopping with **iterative descent**: starting from an initial point $x^{(0)}$, compute a descent direction $d^{(k)}$, take a step $x^{(k+1)} = x^{(k)} + \alpha^{(k)} d^{(k)}$, and repeat until convergence. The choice of direction and step-size distinguishes gradient descent (Lecture 13), interior-point (Lecture 18), and SQP (Lecture 19).

---

## Transportation Problems as NLPs

To make the taxonomy concrete, here are four transportation planning problems — each non-linear for a different structural reason. These motivate the four application lectures (20–22).

### Example 1 — Signal Timing at a Chennai Intersection (Convex QP)

CMRL wants to set green-time splits $(x_1, x_2)$ at a junction to minimise total vehicle delay. Webster's delay formula gives:

$$D_i(x_i) = \frac{C(1 - x_i/C)^2}{2(1 - \lambda_i x_i / C)} + \frac{\lambda_i^2}{2\mu_i(1-\lambda_i)}$$

where $C$ is cycle length, $\lambda_i$ arrival rate, $\mu_i$ saturation flow. The total delay $D_1(x_1) + D_2(x_2)$ is a **convex function** of $(x_1, x_2)$ subject to $x_1 + x_2 \leq C$ (linear). This is a **convex constrained NLP** — local minimum = global minimum; KKT conditions are sufficient.

### Example 2 — Facility Location for a Kochi Freight Hub (Non-Linear, Convex)

A logistics firm wants to locate a transshipment hub at coordinates $(x, y)$ to minimise total weighted distance to $n$ customer locations $(a_k, b_k)$ with demand weights $w_k$:

$$\min_{x,y} \quad f(x,y) = \sum_{k=1}^{n} w_k \sqrt{(x - a_k)^2 + (y - b_k)^2}$$

The Euclidean distance makes this non-linear but still **convex** (sum of convex functions). We study this Weber problem in Lecture 20.

### Example 3 — Traffic Equilibrium on the Chennai–Bengaluru Corridor (Non-Linear, Convex)

The Beckmann formulation of Wardrop's user equilibrium assigns traffic volumes $v_a$ to roads to solve:

$$\min_{v} \quad \sum_{a \in \mathcal{A}} \int_0^{v_a} t_a(u)\, du$$

where $t_a(\cdot)$ is the BPR travel-time function. The integral of BPR is a degree-5 polynomial — non-linear but **convex** on the feasible flow region. Lecture 21 develops this.

### Example 4 — Vehicle Routing in Mumbai Last-Mile Delivery (Non-Convex, Integer)

Assigning $n$ delivery stops to $k$ vehicles and sequencing each vehicle's route is an **integer non-linear program** — the objective (total travel distance) is non-linear in route assignment, and the decision variables include binary routing decisions. This is the TSP/VRP family; **non-convex**, multiple local optima guaranteed. Exact NLP methods handle small instances or continuous relaxations. Lecture 22 addresses this.

---

## In-Class Exercises

### Exercise 1

BMTC is planning bus frequencies $x_1, x_2$ (buses/hour) on two Bengaluru corridors. Passenger demand on each corridor increases with service frequency, but at a diminishing rate. Consequently, the total revenue is given by: $R(x_1, x_2) = 800\sqrt{x_1} + 600\sqrt{x_2}$. Notably, BMTC has a fleet availability constraint that limits the total number of buses deployed to 20 per hour. In addition, to maintain a minimum level of service, each corridor must receive at least 3 buses per hour. Given, operating cost of ₹1,200 per bus-hour, 

1. Write the NLP in standard form.
2. Classify the problem — constrained or unconstrained, linear or non-linear, convex or non-convex.

### Exercise 2

A Delhivery cross-dock facility in Guwahati processes freight arriving at rate $\lambda$ (tonnes/hour). The variable processing cost increases with throughput and is given by: $C(\lambda) = 50\lambda + 2\lambda^2$. In addition, the facility incurs a fixed operating cost of ₹8,000 per hour. The facility earns revenue of ₹200 for every tonne of freight processed. To satisfy the service-level agreement, the facility must process at least 30 tonnes per hour, however, due to equipment capacity constraints, throughput cannot exceed 80 tonnes per hour.

1. Write the NLP in standard form.
2. Classify the problem — constrained or unconstrained, linear or non-linear, convex or non-convex.

---

## Take-Away Exercises

### Exercise 1

A NHAI corridor study estimates travel time on Chandigarh-Ambala link using the BPR function:

$$t(v) = 15 \left[1 + 0.15 \left(\frac{v}{1{,}200}\right)^4\right] \quad \text{(minutes)}$$

where $v$ is traffic volume (vehicles/hour) and capacity is $c = 1{,}200$ veh/hr.

1. Compute $t(600)$, $t(1{,}200)$, and $t(2{,}400)$. Comment on how travel time responds to volume near and above capacity.
2. Differentiate $t(v)$ with respect to $v$ and evaluate $t'(1{,}200)$. Interpret the result: what does this derivative tell NHAI about marginal congestion at capacity?
3. Is $t(v)$ a convex function of $v$ for $v \geq 0$? Justify by computing $t''(v)$ and checking its sign.

### Exercise 2

The traffic volume $v$ on a Mumbai expressway link is assigned to minimise the Beckmann objective:

$$B(v) = \int_0^{v} t_0 \left[1 + 0.15\left(\frac{u}{c}\right)^4\right] du $$

with $t_0 = 20$ min, $c = 2{,}000$ veh/hr, and the constraint $0 \leq v \leq 3{,}000$.

1. Is $B(v)$ convex on $[0, 3000]$? Show using $B''(v)$.
2. The demand constraint requires $v \geq 1{,}500$. Write the constrained NLP in standard form ($g(v) \leq 0$).

---

## Further Reading

- Bazaraa, M.S., Sherali, H.D., and Shetty, C.M. (2006). *Nonlinear Programming: Theory and Algorithms* (3rd ed.). Wiley — Chapters 1–2 (problem classes, optimality conditions overview).
- Nocedal, J. and Wright, S.J. (2006). *Numerical Optimization* (2nd ed.). Springer — Chapter 1 (introduction), Chapter 2 (fundamentals of unconstrained optimisation).
- Wardrop, J.G. (1952). Some theoretical aspects of road traffic research. *Proceedings of the Institute of Civil Engineers*, 1(3), 325–362. — Origin of the traffic equilibrium conditions.
- BPR (1964). *Traffic Assignment Manual*. US Bureau of Public Roads, Washington DC. — Source of the BPR travel-time function.
- SciPy documentation: `scipy.optimize.minimize` — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)